# Graph Construction and Querying with Cypher

In [ ]:
import getpass

# configuration llm
google_llm_model = "gemma-4-31b-it"
# google_embeder_model = "gemini-embedding-2"
google_api_key = getpass.getpass("Enter your Google AI API key: ")

# configuration embedder
hugging_face_embeder_model = "google/embeddinggemma-300m"
hugging_face_key = getpass.getpass("Enter your Hugging Face API key: ")

# configuration databasis
neo4j_uri="neo4j+s://312151bc.databases.neo4j.io"
neo4j_username="312151bc"
neo4j_password = getpass.getpass("Enter your Neo4j password: ")
neo4j_database="312151bc"

## Check Ontology

In [2]:
from pymprev.ontology import Ontology

ontology = Ontology()
ontology.description

'This ontology reflects relevant aspects in the development, maintenance or deployment ofproducts and components based on or using Artificial Inteligence and Machine Learning,with focus on recomendation compliance and safety, in the context of civil aviation.\n\nThe node schemas (properties and descriptions): \n\tAiml_element:\n\t * description: elements or components. systems and subsystems, models, tools and products based on or using AI/ML: Neural Nets, Models, etc\n\t * properties: definition, purpose, name, description\n\tAiml_resource:\n\t * description: training sets, generators, accelerators (GPU/FPGA), libraries, parameters, hyperparameters\n\t * properties: definition, name, description\n\tAiml_lifecycle_aspect:\n\t * description: Aspects related to AI/ML development and maintenance: implementation, training, validation, deployment, etc\n\t * properties: key_deliverable, definition, name, description\n\tRisk_factor:\n\t * description: Risks system or component may be subject 

## Connection and File Ingestion

In [3]:
from pymprev.connection import connect_llm, connect_embeder, connect_database

/home/fredson-aguiar/Downloads/venv-mprev/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# connect services
llm = connect_llm(
    google_api_key,
    google_llm_model)

embedder = connect_embeder(
    hugging_face_key,
    hugging_face_embeder_model)

graph = connect_database(
    neo4j_uri,
    neo4j_database,
    neo4j_username,
    # sanitize=True,
    # enhanced_schema=True,
    neo4j_password)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1929.94it/s]


In [5]:
from pymprev.util import GraphRAG

graphRAG = GraphRAG(llm, embedder, graph, ontology)

### Documents Ingestion

In [6]:
# graphRAG.ingest_documents("/home/fredson-aguiar/Downloads/resources/")

In [ ]:
graphRAG.unify_nodes_by_similarity(
    high_threshold=0.995, low_threshold=0.95, text_threshold=0.95)

total merged nodes of type Aiml_element: 227
total merged nodes of type Aiml_resource: 72
total merged nodes of type Aiml_lifecycle_aspect: 117
total merged nodes of type Risk_factor: 88
total merged nodes of type Risk_mitigation_strategy: 124


In [9]:
graphRAG.add_knn_similarity_relations(score_threshold=0.95, top_k=5)

finding similar neighbors per type: Aiml_element
finding similar neighbors per type: Aiml_resource
finding similar neighbors per type: Aiml_lifecycle_aspect
finding similar neighbors per type: Risk_factor
finding similar neighbors per type: Risk_mitigation_strategy


### Create Bottom-up Clusterization

In [ ]:
graphRAG.run_leiden_clustering(
    max_cluster_size=15, resolution=0.5, ontology_weight=1.0)

Adding Leiden cluster structure to Database
Creating bottom-up cluster summarization: level 7


100%|██████████| 2/2 [00:07<00:00,  3.73s/it]


Creating bottom-up cluster summarization: level 6


100%|██████████| 13/13 [00:50<00:00,  3.88s/it]


Creating bottom-up cluster summarization: level 5


...

100%|██████████| 313/313 [23:10<00:00,  4.44s/it]


Creating bottom-up cluster summarization: level 1


100%|██████████| 80/80 [06:46<00:00,  5.08s/it]


Creating bottom-up cluster summarization: level 0


100%|██████████| 17/17 [01:22<00:00,  4.83s/it]


In [ ]:
cluster_ids = graphRAG.get_cluster_ids_by_level(0)
for cluster_id in cluster_ids:
    n_childnodes = len(graphRAG.get_children_entities(cluster_id))
    n_subclusters = len(graphRAG.get_children_clusters(cluster_id))
    name, summary, level = graphRAG.get_cluster_details(cluster_id)

    print(f"cluster {cluster_id}, {n_subclusters+n_childnodes} children, '{name}': {summary}")

cluster 0, 8 children, 'AI Governance, Safety, and Risk Management': This cluster synthesizes a multi-dimensional framework for the deployment of trustworthy AI/ML systems in civil aviation, integrating technical, regulatory, and ethical governance. It encompasses the full AI/ML lifecycle—from foundational infrastructure (HPC, cloud, and data governance) and resource management to the operational integration of human-AI teaming and systemic safety. The framework addresses a broad spectrum of risk factors, including technical vulnerabilities (adversarial attacks, model opacity, and distribution drift), human-centric risks (automation bias and complacency), and socio-environmental impacts (algorithmic bias, human rights violations, and ecological footprints). To ensure safety-critical reliability and regulatory compliance (e.g., AI Act, GDPR), the cluster integrates rigorous risk mitigation strategies, such as eXplainable AI (XAI), formal verification, human-in-the-loop (HITL) oversight,

In [26]:
cluster_id = 7

childnodes = graphRAG.get_children_entities(cluster_id)
subclusters = graphRAG.get_children_clusters(cluster_id)

for r in childnodes:
    text = graphRAG._node_as_text(r['node_id'], r['labels'], r['name'], r['description'])
    print(text)

for r in subclusters:
    text = graphRAG._cluster_as_text(r['child_id'], r['name'], r['summary'])
    print(text)

Cluster ID: 78 
	* name: Speech and Voice Processing Systems
	* description: This cluster focuses on the development and deployment of AI/ML elements specialized in human language interpretation, encompassing Speech-to-Text (STT), Text-to-Speech (TTS), and voice recognition systems. It integrates cognitive systems for acoustic modeling and identity validation (voiceprints) using tools like Speechbrain and Unispeech. Within the context of civil aviation, these technologies are applied to Automatic Speech Recognition (ASR) for transcribing Air Traffic Control (ATC) communications to reduce controller workload, utilizing architectures such as DeepSpeech, Kaldi, and WaveNet. The cluster also addresses critical risk factors, including adversarial attacks and operational challenges like multilingual accents and lexical ambiguities, aligning with the ontology's focus on safety, compliance, and risk mitigation in aviation environments.
Cluster ID: 79 
	* name: ATC Speech-to-Text Systems and Ri

In [ ]:
cluster_id = 1427

childnodes = graphRAG.get_children_entities(cluster_id)
subclusters = graphRAG.get_children_clusters(cluster_id)

for r in childnodes:
    text = graphRAG._node_as_text(r['node_id'], r['labels'], r['name'], r['description'])
    print(text)

for r in subclusters:
    text = graphRAG._cluster_as_text(r['child_id'], r['name'], r['summary'])
    print(text)

Node ID/Type: Validation And Certification Process/['Aiml_lifecycle_aspect'] 
	* name: None
	* description: The process of validating and certifying the SHM system based on application intent and criticality
Node ID/Type: Validation Program/['Aiml_lifecycle_aspect'] 
	* name: None
	* description: Program to establish that the SHM system is effective in detecting damage
Node ID/Type: Shm Validation Processes/['Aiml_lifecycle_aspect'] 
	* name: None
	* description: Processes to validate SHM solutions, including objective evaluation of skills, automation, and human error, and performance assessment


### Reseting Databasis, Embeddings or Removing document

In [8]:
# # reset complete databasis
# graphRAG.reset_databasis(graph)

In [9]:
# # reset embeddings
# graphRAG.reset_chunk_embeddings(10)
# graphRAG.reset_node_embeddings(10)

In [ ]:
# # delete clusters
# graphRAG.delete_clustering()

In [10]:
# # removing single file and instances created from it
# filename = "/home/fredson-aguiar/Downloads/resources/Machine Learning Safety and Applicability/ARP6983 v8_260306.pdf"
# graphRAG.remove_document(graph, filename=filename)